In [1]:
# Data handling
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re

# Download NLP resources
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
# Load CSV file
df = pd.read_csv("truck_data_sample.csv")

# Display first rows
df.head()

,id,brand,model,year,variant_name,trim,segment,body_type,cab_type,axle_config,...,price_eur,price_usd,features,is_ev,is_hydrogen,is_cng,safety_rating,source_url,data_source,last_updated
0,950,DAF,CF-series,NaN,DAF CF-series FAG 9.2TD (310 Hp),FAG 9.2TD (310 Hp),Medium,Chassis,2-berth with 1 sleeping,6x2,...,NaN,NaN,"[""Max power at: 2200 min-1"", ""Max torque at: 1...",False,False,False,NaN,NaN,car2db,2026-04-03T13:47:08.787610
1,958,DAF,CF-series,NaN,DAF CF-series FAG 9.2TD (360 Hp),FAG 9.2TD (360 Hp),Medium,Chassis,2-berth with 1 sleeping,6x2,...,NaN,NaN,"[""Max power at: 2200 min-1"", ""Max torque at: 1...",False,False,False,NaN,NaN,car2db,2026-04-03T13:47:08.787610
2,2854,DAF,CF85.430,NaN,DAF CF85.430 FTT 12.6TD (430 Hp),FTT 12.6TD (430 Hp),Heavy,NaN,2- places with 2 beds,6x4,...,NaN,NaN,"[""Cabin type: 2- places with 2 beds"", ""Engine ...",False,False,False,NaN,https://truck-data.com/en/truck/DAF/CF85.430/F...,truck-data.com,2026-04-04T14:15:24.643985
3,4464,Fiat,Doblo Cargo,NaN,Fiat Doblo Cargo 1.3 JTD MultiJet (70 Hp),1.3 JTD MultiJet (70 Hp),Light,Van,NaN,FWD,...,NaN,NaN,"[""ECO standard: EURO II"", ""Maximum power at rp...",False,False,False,NaN,https://truck-data.com/en/light-truck/Fiat/Dob...,truck-data.com,2026-04-04T14:15:24.643985
4,4465,Fiat,Doblo Cargo,NaN,Fiat Doblo Cargo 1.4 8V (77 Hp),1.4 8V (77 Hp),Light,Van,NaN,FWD,...,NaN,NaN,"[""ECO standard: EURO IV"", ""Maximum power at rp...",False,False,False,NaN,https://truck-data.com/en/light-truck/Fiat/Dob...,truck-data.com,2026-04-04T14:15:24.643985


In [3]:
# Dataset information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 50 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         30 non-null     int64  
 1   brand                      30 non-null     str    
 2   model                      30 non-null     str    
 3   year                       0 non-null      float64
 4   variant_name               30 non-null     str    
 5   trim                       30 non-null     str    
 6   segment                    30 non-null     str    
 7   body_type                  20 non-null     str    
 8   cab_type                   10 non-null     str    
 9   axle_config                25 non-null     str    
 10  engine_type                14 non-null     str    
 11  fuel_type                  27 non-null     str    
 12  cubic_capacity_cc          29 non-null     float64
 13  cylinders                  30 non-null     int64  
 14  aspirat

In [5]:


df.head()

,id,brand,model,year,variant_name,trim,segment,body_type,cab_type,axle_config,...,price_eur,price_usd,features,is_ev,is_hydrogen,is_cng,safety_rating,source_url,data_source,last_updated
0,950,DAF,CF-series,NaN,DAF CF-series FAG 9.2TD (310 Hp),FAG 9.2TD (310 Hp),Medium,Chassis,2-berth with 1 sleeping,6x2,...,NaN,NaN,"[""Max power at: 2200 min-1"", ""Max torque at: 1...",False,False,False,NaN,NaN,car2db,2026-04-03T13:47:08.787610
1,958,DAF,CF-series,NaN,DAF CF-series FAG 9.2TD (360 Hp),FAG 9.2TD (360 Hp),Medium,Chassis,2-berth with 1 sleeping,6x2,...,NaN,NaN,"[""Max power at: 2200 min-1"", ""Max torque at: 1...",False,False,False,NaN,NaN,car2db,2026-04-03T13:47:08.787610
2,2854,DAF,CF85.430,NaN,DAF CF85.430 FTT 12.6TD (430 Hp),FTT 12.6TD (430 Hp),Heavy,NaN,2- places with 2 beds,6x4,...,NaN,NaN,"[""Cabin type: 2- places with 2 beds"", ""Engine ...",False,False,False,NaN,https://truck-data.com/en/truck/DAF/CF85.430/F...,truck-data.com,2026-04-04T14:15:24.643985
3,4464,Fiat,Doblo Cargo,NaN,Fiat Doblo Cargo 1.3 JTD MultiJet (70 Hp),1.3 JTD MultiJet (70 Hp),Light,Van,NaN,FWD,...,NaN,NaN,"[""ECO standard: EURO II"", ""Maximum power at rp...",False,False,False,NaN,https://truck-data.com/en/light-truck/Fiat/Dob...,truck-data.com,2026-04-04T14:15:24.643985
4,4465,Fiat,Doblo Cargo,NaN,Fiat Doblo Cargo 1.4 8V (77 Hp),1.4 8V (77 Hp),Light,Van,NaN,FWD,...,NaN,NaN,"[""ECO standard: EURO IV"", ""Maximum power at rp...",False,False,False,NaN,https://truck-data.com/en/light-truck/Fiat/Dob...,truck-data.com,2026-04-04T14:15:24.643985


In [6]:
# Find text columns
text_columns = df.select_dtypes(include=['object']).columns

text_columns

/var/folders/88/w4w1n8l12kd_z42_6mrnndmw0000gn/T/ipykernel_29909/1518534922.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include=['object']).columns


Index(['brand', 'model', 'variant_name', 'trim', 'segment', 'body_type',
       'cab_type', 'axle_config', 'engine_type', 'fuel_type', 'aspiration',
       'emission_standard', 'transmission_type', 'transmission_model',
       'features', 'source_url', 'data_source', 'last_updated'],
      dtype='str')

In [7]:
df.head()

,id,brand,model,year,variant_name,trim,segment,body_type,cab_type,axle_config,...,price_eur,price_usd,features,is_ev,is_hydrogen,is_cng,safety_rating,source_url,data_source,last_updated
0,950,DAF,CF-series,NaN,DAF CF-series FAG 9.2TD (310 Hp),FAG 9.2TD (310 Hp),Medium,Chassis,2-berth with 1 sleeping,6x2,...,NaN,NaN,"[""Max power at: 2200 min-1"", ""Max torque at: 1...",False,False,False,NaN,NaN,car2db,2026-04-03T13:47:08.787610
1,958,DAF,CF-series,NaN,DAF CF-series FAG 9.2TD (360 Hp),FAG 9.2TD (360 Hp),Medium,Chassis,2-berth with 1 sleeping,6x2,...,NaN,NaN,"[""Max power at: 2200 min-1"", ""Max torque at: 1...",False,False,False,NaN,NaN,car2db,2026-04-03T13:47:08.787610
2,2854,DAF,CF85.430,NaN,DAF CF85.430 FTT 12.6TD (430 Hp),FTT 12.6TD (430 Hp),Heavy,NaN,2- places with 2 beds,6x4,...,NaN,NaN,"[""Cabin type: 2- places with 2 beds"", ""Engine ...",False,False,False,NaN,https://truck-data.com/en/truck/DAF/CF85.430/F...,truck-data.com,2026-04-04T14:15:24.643985
3,4464,Fiat,Doblo Cargo,NaN,Fiat Doblo Cargo 1.3 JTD MultiJet (70 Hp),1.3 JTD MultiJet (70 Hp),Light,Van,NaN,FWD,...,NaN,NaN,"[""ECO standard: EURO II"", ""Maximum power at rp...",False,False,False,NaN,https://truck-data.com/en/light-truck/Fiat/Dob...,truck-data.com,2026-04-04T14:15:24.643985
4,4465,Fiat,Doblo Cargo,NaN,Fiat Doblo Cargo 1.4 8V (77 Hp),1.4 8V (77 Hp),Light,Van,NaN,FWD,...,NaN,NaN,"[""ECO standard: EURO IV"", ""Maximum power at rp...",False,False,False,NaN,https://truck-data.com/en/light-truck/Fiat/Dob...,truck-data.com,2026-04-04T14:15:24.643985


In [9]:
df.columns

Index(['id', 'brand', 'model', 'year', 'variant_name', 'trim', 'segment',
       'body_type', 'cab_type', 'axle_config', 'engine_type', 'fuel_type',
       'cubic_capacity_cc', 'cylinders', 'aspiration', 'power_hp', 'power_kw',
       'torque_nm', 'emission_standard', 'transmission_type',
       'transmission_model', 'gears', 'gvw_kg', 'gcw_kg', 'kerb_weight_kg',
       'payload_capacity_kg', 'max_towing_kg', 'length_mm', 'width_mm',
       'height_mm', 'wheelbase_mm', 'top_speed_kmh', 'gradeability_pct',
       'fuel_consumption_combined', 'co2_emissions_gkm', 'adblue_tank_l',
       'fuel_tank_l', 'battery_capacity_kwh', 'range_km', 'charging_power_kw',
       'price_eur', 'price_usd', 'features', 'is_ev', 'is_hydrogen', 'is_cng',
       'safety_rating', 'source_url', 'data_source', 'last_updated',
       'combined_text'],
      dtype='str')

In [10]:
df.to_csv(
    "truck_data_clustered.csv",
    index=False
)

print("Saved successfully")

Saved successfully
